# Baseline:

In [1]:
import pymupdf4llm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import MarkdownHeaderTextSplitter 
import faiss
import numpy as np

c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Retriever:

In [2]:
md = pymupdf4llm.to_markdown(r".\data\MARTA.pdf")

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x82 in position 87: invalid start byte
Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\alaab\miniconda3\envs\RAG_P

In [3]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

text_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
sections = text_splitter.split_text(md)

## Embedding and Indexation:

In [4]:
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

<All keys matched successfully>


In [ ]:
texts = [s.page_content for s in sections]
metadatas = [s.metadata for s in sections]
embeddings = model.encode(texts)

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

## Main

In [ ]:
import ollama

def query_pipeline(user_question, k=10):
    question_embedding = model.encode([user_question])
    
    distances, indices = index.search(np.array(question_embedding).astype('float32'), k)
    
    context = ""
    for idx in indices[0]:
        section_name = metadatas[idx].get('Header 1', 'General')
        context += f"\n[Source: {section_name}]\n{texts[idx]}\n"

    prompt = f"""
    You are an AI research assistant.

    Answer the question using ONLY the provided context.
    If the answer is not present in the context, say:
    "The paper does not provide this information.

    Context: {context}
    Question: {user_question}
    Answer:"""

    response = ollama.generate(model='llama3', prompt=prompt)
    return response['response']

print(query_pipeline("How was the problem formulated"))

# improved

In [ ]:
import pymupdf4llm
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
import faiss
import numpy as np
import ollama

## Retriever:

In [ ]:
pdf_path = r"data\MARTA.pdf"
paper_title = pdf_path.split("\\")[-1]

md = pymupdf4llm.to_markdown(pdf_path)

In [ ]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
sections = header_splitter.split_text(md)

In [ ]:
chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = []
metadatas = []

for section in sections:
    section_text = section.page_content
    section_meta = section.metadata

    split_chunks = chunk_splitter.split_text(section_text)

    for i, chunk in enumerate(split_chunks):

        metadata = {
            "paper": paper_title,
            "section": section_meta.get("Header 1", "Unknown"),
            "subsection": section_meta.get("Header 2", ""),
            "chunk_id": len(chunks)
        }

        chunks.append(chunk)
        metadatas.append(metadata)

## Embeding and Indexation:

In [ ]:
model = SentenceTransformer(
    "nomic-ai/nomic-embed-text-v1.5",
    trust_remote_code=True
)

embeddings = model.encode(
    chunks,
    normalize_embeddings=True
)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(np.array(embeddings).astype("float32"))

In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-base")

In [ ]:
def retrieve_chunks(question, k=15):

    question_embedding = model.encode(
        [question],
        normalize_embeddings=True
    )

    distances, indices = index.search(
        np.array(question_embedding).astype("float32"),
        k
    )

    candidate_chunks = [chunks[i] for i in indices[0]]
    candidate_metadata = [metadatas[i] for i in indices[0]]

    pairs = [[question, chunk] for chunk in candidate_chunks]
    scores = reranker.predict(pairs)

    reranked = sorted(
        zip(candidate_chunks, candidate_metadata, scores),
        key=lambda x: x[2],
        reverse=True
    )

    return reranked[:5]

In [ ]:
def build_context(retrieved):

    context = ""

    for chunk, metadata, score in retrieved:

        context += f"""
[Paper: {metadata['paper']}]
[Section: {metadata['section']}]
[Subsection: {metadata['subsection']}]

{chunk}

---
"""

    return context


def query_pipeline(user_question):

    retrieved = retrieve_chunks(user_question)

    context = build_context(retrieved)

    prompt = f"""
You are an AI research assistant.

Answer the question using ONLY the provided context.
If the answer is not present in the context, say:
"The paper does not provide this information."

Always cite sources in the format:
(Paper: <paper>, Section: <section>)

Context:
{context}

Question:
{user_question}

Answer:
"""

    response = ollama.generate(
        model="llama3",
        prompt=prompt
    )

    return response["response"]


question = "How was the problem formulated"

answer = query_pipeline(question)

print("\nQUESTION:\n", question)
print("\nANSWER:\n", answer)

In [3]:
md = pymupdf4llm.to_markdown("data\Human-Robot Collaboration.pdf")

<>:1: SyntaxWarning: invalid escape sequence '\H'
<>:1: SyntaxWarning: invalid escape sequence '\H'
C:\Users\alaab\AppData\Local\Temp\ipykernel_13732\1069797706.py:1: SyntaxWarning: invalid escape sequence '\H'
  md = pymupdf4llm.to_markdown("data\Human-Robot Collaboration.pdf")
Exception in thread Thread-22 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x82 in position 87: invalid start byte
Exception in thread Thread-24 (_readerthread):
Traceback (most recent call last):
  File "c:\U

In [ ]:
import gradio as gr

def answer_from_pdf(pdf_file, question):
    """
    pdf_file: a temporary file path or file-like object (depends on type)
    question: string from the textbox
    Returns: string (answer or extracted text)
    """
    md = pymupdf4llm.to_markdown(pdf_file)
    return f"Question: {pdf_file.name}\n\nAnswer:\n{md[:500]}"

demo = gr.Interface(
    fn=answer_from_pdf,
    inputs=[
        gr.File(label="Upload your PDF", file_types=[".pdf"]),
        gr.Textbox(label="Ask a question", lines=2, placeholder="e.g., What is the summary?")
    ],
    outputs=gr.Textbox(label="Answer", lines=10),
    title="Research paper assistant",
    description="Upload a PDF and ask a question about its content."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Exception in thread Thread-47 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\subprocess.py", line 1599, in _readerthread
    buffer.append(fh.read())
                  ^^^^^^^^^
  File "<frozen codecs>", line 322, in decode
UnicodeDecodeError: 'utf-8' codec can't decode byte 0x82 in position 87: invalid start byte
Exception in thread Thread-49 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\alaab\miniconda3\envs\RAG_Project\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\alaab\miniconda3\envs\RAG

In [ ]:
from retriever import chunking_baseline, chunking_improved
from embedding import embedding_baseline, embedding_improved, retrieve_chunks, build_context
from sentence_transformers import SentenceTransformer
import ollama
import numpy as np


def query_pipeline_improved(user_question, md, pdf_file):

    chunks, metadatas= chunking_improved(md, pdf_file)

    index= embedding_improved(chunks)

    retrieved = retrieve_chunks(index, chunks, metadatas, user_question)

    context = build_context(retrieved)

    prompt = f"""
You are an AI research assistant.

Answer the question using ONLY the provided context.
If the answer is not present in the context, say:
"The paper does not provide this information."

Always cite sources in the format:
(Paper: <paper>, Section: <section>)

Context:
{context}

Question:
{user_question}

Answer:
"""

    response = ollama.generate(
        model="llama3",
        prompt=prompt
    )

    return response["response"]